In [8]:
# %%
# STEP 1 — Load EPC dataset and filter target archetypes
# Purpose: Select all SemiDetached_2F AND Detached_2F buildings

"""
SCRIPT OVERVIEW
---------------
This script processes EPC-derived building geometry data and updates .geo model files
for multiple building archetypes.

Key updates:
- Now includes BOTH "SemiDetached_2F" and "Detached_2F"
- Iterates through each building and applies geometry transformations:
    - Footprint scaling
    - Window width/placement adjustments
    - Window height adjustments
    - Roof geometry updates
- Writes updated geometry back into the original .geo files (in-place)

Intended use:
- Batch processing of building models derived from EPC data
- Supports scalable geometry updates for multiple archetypes

Requirements:
- Correct folder structure in baseline_models directory
- EPC dataset with required geometric parameters
"""

import pandas as pd

epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"

df = pd.read_csv(epc_path)

# UPDATED: include both archetypes
target_archetypes = ["SemiDetached_2F", "Detached_2F"]
target_df = df[df["ARCHETYPE"].isin(target_archetypes)].copy()

print(f"Total target buildings: {len(target_df)}")
print(target_df["ARCHETYPE"].value_counts())

Total target buildings: 54
ARCHETYPE
Detached_2F        32
SemiDetached_2F    22
Name: count, dtype: int64


In [9]:
# %%
# STEP 2 — Define base model directory
# Purpose: Point to all building model folders

from pathlib import Path

base_models_dir = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

In [10]:
# %%
# STEP 3 — File utilities
# Purpose: Reusable file reader

def read_file(path):
    with open(path, "r") as f:
        return f.readlines()

In [11]:
# %%
# STEP 4 — Vertex parser (unchanged)
# Purpose: Preserve geometry integrity

def parse_vertices(lines):
    vertices = {}
    vertex_line_map = {}

    for i, line in enumerate(lines):
        if line.strip().startswith("*vertex"):

            parts = line.split(",")

            vid = int(parts[-1].split("#")[-1].strip())

            x = float(parts[1].strip())
            y = float(parts[2].strip())
            z = float(parts[3].split("#")[0].strip())

            vertices[vid] = [x, y, z]
            vertex_line_map[vid] = i

    return vertices, vertex_line_map

In [12]:
# %%
# STEP 5 — Geometry transformations (unchanged)
# Purpose: Apply all geometry logic

def apply_footprint(zone, v, width, length):

    if zone in ["flr0", "flr1"]:
        v[2][0] = width
        v[3][0] = width; v[3][1] = length
        v[4][1] = length

        v[6][0] = width
        v[7][0] = width; v[7][1] = length
        v[8][1] = length

    elif zone == "attic":
        v[2][0] = width
        v[3][0] = width; v[3][1] = length
        v[4][1] = length

    return v


def apply_window_width(zone, v, w, building_width):

    if zone == "flr0":

        v[13][1] = v[14][1] + w
        v[16][1] = v[15][1] + w

        for vid in range(17, 25):
            v[vid][0] = building_width

        v[17][1] = 0.5
        v[20][1] = 0.5

        v[18][1] = v[17][1] + w
        v[19][1] = v[20][1] + w

        v[21][1] = v[18][1] + 0.5
        v[24][1] = v[19][1] + 0.5

        v[22][1] = v[21][1] + w
        v[23][1] = v[24][1] + w

    elif zone == "flr1":

        v[9][1]  = v[10][1] + w
        v[12][1] = v[11][1] + w

        v[14][1] = v[9][1] + 0.5
        v[15][1] = v[12][1] + 0.5

        v[13][1] = v[14][1] + w
        v[16][1] = v[15][1] + w

        for vid in range(17, 25):
            v[vid][0] = building_width

        v[21][1] = 0.5
        v[24][1] = 0.5

        v[22][1] = v[21][1] + w
        v[23][1] = v[21][1] + w

        v[17][1] = v[22][1] + 0.5
        v[20][1] = v[23][1] + 0.5

        v[18][1] = v[17][1] + w
        v[19][1] = v[20][1] + w

    return v


def apply_window_height(zone, v, window_height):

    if zone == "flr0":
        floor_z = 0.0
        targets = [16, 15, 23, 24, 19, 20]

    elif zone == "flr1":
        floor_z = v[1][2]
        targets = [16, 15, 12, 11, 23, 24, 19, 20]

    else:
        return v

    sill = floor_z + 0.75
    top  = sill + window_height

    for vid in targets:
        v[vid][2] = top

    return v


def apply_roof(zone, v, width, ridge_height):

    if zone != "attic":
        return v

    half_width = width / 2.0

    v[5][0] = half_width
    v[6][0] = half_width

    v[5][1] = v[1][1]
    v[6][1] = v[4][1]

    v[5][2] = ridge_height
    v[6][2] = ridge_height

    return v

In [13]:
# %%
# STEP 6 — Rebuild function (unchanged)
# Purpose: Write updated geometry back into files

def rebuild(lines, vertices, vertex_line_map):

    new_lines = lines.copy()

    for vid, coords in vertices.items():
        i = vertex_line_map[vid]
        x, y, z = coords

        new_lines[i] = f"*vertex,{x:.5f},{y:.5f},{z:.5f}  # {vid}\n"

    return new_lines

In [14]:
# %%
# STEP 7 — Main loop over ALL buildings (UPDATED CORE LOGIC)
# Purpose: Apply transformations per building and overwrite files in-place

zones = ["attic", "flr0", "flr1"]

for idx, row in target_df.iterrows():

    building_id = str(row["BUILDING_REFERENCE_NUMBER"])
    archetype   = row["ARCHETYPE"]

    # Geometry parameters (per building)
    building_width  = float(row["WALL_WIDTH"])
    building_length = float(row["WALL_LENGTH"])
    window_height   = float(row["WINDOW_H"])
    window_width    = float(row["WINDOW_W"])
    ridge_height    = float(row["ROOF_RIDGE_HEIGHT"])

    # Construct path to this building's zones folder
    zones_dir = base_models_dir / building_id / archetype / "zones"

    if not zones_dir.exists():
        print(f"⚠️ Missing zones folder for {building_id} ({archetype})")
        continue

    print(f"Processing building {building_id} ({archetype})")

    for zone in zones:

        geo_path = zones_dir / f"{zone}.geo"

        if not geo_path.exists():
            print(f"  ⚠️ Missing file: {geo_path}")
            continue

        # Read
        lines = read_file(geo_path)

        # Parse
        v, vertex_line_map = parse_vertices(lines)

        # Transform
        v = apply_footprint(zone, v, building_width, building_length)
        v = apply_window_width(zone, v, window_width, building_width)
        v = apply_window_height(zone, v, window_height)
        v = apply_roof(zone, v, building_width, ridge_height)

        # Rebuild
        new_lines = rebuild(lines, v, vertex_line_map)

        # OVERWRITE ORIGINAL FILE
        with open(geo_path, "w") as f:
            f.writelines(new_lines)

    print(f"  ✅ Completed {building_id}")

print("🎯 ALL SemiDetached_2F AND Detached_2F MODELS UPDATED IN-PLACE")

Processing building 1001829067 (SemiDetached_2F)
  ✅ Completed 1001829067
Processing building 1001951858 (Detached_2F)
  ✅ Completed 1001951858
Processing building 1234761001 (Detached_2F)
  ✅ Completed 1234761001
Processing building 1001664929 (SemiDetached_2F)
  ✅ Completed 1001664929
Processing building 1001829059 (SemiDetached_2F)
  ✅ Completed 1001829059
Processing building 1001664925 (Detached_2F)
  ✅ Completed 1001664925
Processing building 1001870854 (SemiDetached_2F)
  ✅ Completed 1001870854
Processing building 1002143352 (Detached_2F)
  ✅ Completed 1002143352
Processing building 1002143353 (Detached_2F)
  ✅ Completed 1002143353
Processing building 1001829085 (SemiDetached_2F)
  ✅ Completed 1001829085
Processing building 1001829058 (SemiDetached_2F)
  ✅ Completed 1001829058
Processing building 1234806523 (Detached_2F)
  ✅ Completed 1234806523
Processing building 1001664941 (Detached_2F)
  ✅ Completed 1001664941
Processing building 1000336709 (SemiDetached_2F)
  ✅ Completed 100